# Parking-capacity — inférence sur une adresse française

Notebook de **démo** : on tape une adresse, le pipeline renvoie une estimation de capacité parking privée + parking public OSM voisin + flags de cohérence.

Pipeline : BAN (géocodage) → cadastre IGN → OSM Overpass → orthophoto IGN BD ORTHO → SegFormer + YOLOv8 + heuristiques géométriques → couche sémantique → couche de cohérence.

**Ce notebook ne nécessite pas de Drive ni de GPU** (CPU suffit pour l'inférence, ~30-60 s par adresse).

## 1. Cloner le dépôt et installer les dépendances

Première exécution : ~3-5 min (téléchargement torch/transformers/ultralytics + installation).

In [ ]:
import os, subprocess, sys

REPO_URL = "https://github.com/chvro12/yolovbis.git"
ROOT = "/content/parking_capacity" if os.path.isdir("/content") else "./parking_capacity"

if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], check=True)

os.chdir(ROOT)
print("Repo prêt :", ROOT)

In [ ]:
# Installation en mode editable. Colab a déjà torch/torchvision, on ajoute le reste.
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", "."], check=True)
print("Dépendances installées.")

## 2. Configuration providers GIS (sans Mapillary, sans secrets)

Le pipeline marche avec uniquement les services publics français (BAN, APICarto, IGN Géoplateforme, OSM). Pas besoin de token.

In [ ]:
# providers.yaml minimal pour Colab (Mapillary désactivé)
providers_yaml = """ign:
  enabled: true
  geoplateforme_wfs_url: https://data.geopf.fr/wfs/ows
  geoplateforme_wms_raster_url: https://data.geopf.fr/wms-r
  use_bdtopo: true
  bdtopo_buildings_typename: BDTOPO_V3:batiment
  bdtopo_roads_typename: BDTOPO_V3:troncon_de_route
  bdtopo_zones_typename: BDTOPO_V3:zone_d_activite_ou_d_interet
  wfs_max_features: 500

osm:
  enabled: true
  overpass_url: https://overpass-api.de/api/interpreter

mapillary:
  enabled: false

fusion:
  access_distance_threshold_m: 40.0
  max_plausible_capacity_slots: 39
"""
open("providers.yaml", "w").write(providers_yaml)
print("providers.yaml écrit")

## 3. Téléchargement du modèle YOLO (COCO yolov8s.pt, ~22 Mo)

Ultralytics télécharge automatiquement à la première utilisation. On force ici pour éviter le délai au premier appel.

In [ ]:
from ultralytics import YOLO
_ = YOLO("yolov8s.pt")  # télécharge si absent
print("YOLOv8s.pt prêt — détection véhicules activée")

## 4. Prédire la capacité parking d'une adresse

Le premier appel charge SegFormer (~700 Mo) depuis HuggingFace, ~1-2 min.

In [ ]:
from parking_capacity.pipeline import process_address, row_to_json_serializable
from pathlib import Path
import tempfile

# 👇 change l'adresse ici
ADDRESS = "6 RUE LEONARD DE VINCI, 91090 LISSES"

chip_tmp = Path(tempfile.mkstemp(suffix=".png")[1])
r = process_address(
    ADDRESS,
    use_vision=True,
    save_chip_path=chip_tmp,
    overpass_delay_s=0.3,
)
d = row_to_json_serializable(r)

## 5. Afficher le résultat

In [ ]:
from IPython.display import Markdown, display
from PIL import Image

def fmt(d):
    return f"""### Résultat — {d.get('input_address')}

- **Coordonnées** : `{d.get('lat')}`, `{d.get('lon')}`
- **Capacité estimée (privée)** : **{d.get('estimated_capacity')}** places (min {d.get('min_capacity')} — max {d.get('max_capacity')})
- **Méthode** : `{d.get('method_used')}`
- **Source primaire** : `{d.get('primary_source')}` (confiance : `{d.get('primary_confidence')}`)
- **Parking public OSM voisin** : `{d.get('nearby_public_capacity_estimate')}` places (source : `{d.get('nearby_public_capacity_source')}`)
- **Véhicules détectés** : {d.get('vehicle_count')} (méthode : `{d.get('vehicle_detection_method')}`)
- **Places marquées détectées** : {d.get('slots_total_count')} (méthode : `{d.get('slot_detection_method')}`)
- **Plafond physique** : {d.get('plausible_capacity_ceiling')} places
- **Cohérence** : `{d.get('consistency_max_severity')}` — needs_review : **{'OUI' if d.get('consistency_needs_review') else 'non'}**
"""

display(Markdown(fmt(d)))

# Flags de cohérence
flags = d.get("consistency_flags") or []
if flags:
    icons = {"high": "🔴", "medium": "🟠", "info": "🔵"}
    lines = ["**Flags de cohérence :**"]
    for f in flags:
        lines.append(f"- {icons.get(f['severity'], '•')} **{f['severity']}** `{f['name']}` — {f['reason']}")
    display(Markdown("\n".join(lines)))
else:
    display(Markdown("_Aucun flag de cohérence — la prédiction est consistante avec les signaux._"))

# Chip orthophoto
if chip_tmp.is_file():
    display(Markdown("**Orthophoto IGN BD ORTHO de la parcelle :**"))
    display(Image.open(chip_tmp))

## 6. (Optionnel) Lancer l'UI Gradio en mode partagé

Donne une URL publique temporaire (valide 72h) pour partager le démo.

In [ ]:
from parking_capacity.ui import build_app

demo = build_app()
demo.queue().launch(share=True)

## 7. (Avancé) Lot d'adresses

Pour évaluer le modèle sur un jeu d'adresses + vérité terrain (ex: `expected_capacity` dans le CSV).

In [ ]:
# Exemple : 6 adresses problématiques de la validation manuelle
csv_text = """address,expected_capacity
"4 RUE JEAN JAURES, 95470 SURVILLIERS",0
"6 RUE LEONARD DE VINCI, 91090 LISSES",55
"112 BOULEVARD ROGER SALENGRO, 69400 VILLEFRANCHE SUR SAONE",10
"9 RUE DU PONT LONDEAU, 61000 CERISE",20
"321 IMPASSE DES CHAMPS, LES TERRES DE L'ARNY, 74350 ALLONZIER LA CAILLE",80
"143 BD LAFAYETTE, 63000 CLERMONT FERRAND",13
"""
open("/tmp/demo_addresses.csv", "w").write(csv_text)

subprocess.run([
    sys.executable, "-m", "parking_capacity.cli", "benchmark-addresses",
    "--input", "/tmp/demo_addresses.csv",
    "--out", "/tmp/demo_results",
    "--overpass-delay", "0.3",
], check=True)

import pandas as pd
df = pd.read_csv("/tmp/demo_results/results.csv")
df[["input_address", "estimated_capacity", "primary_source", "consistency_max_severity"]]